# 🔎 Retrieval-Augmented Generation (RAG) — Hands-on Lab

**ระยะเวลา:** ประมาณ 3 ชั่วโมง
**ระดับ:** ผู้เริ่มต้น–ปานกลาง (มีพื้นฐาน Python และแนวคิด Machine Learning เบื้องต้น)
**สภาพแวดล้อม:** Google Colab (แนะนำ Runtime แบบ GPU: T4)

## 🎯 วัตถุประสงค์การเรียนรู้ (Learning Objectives)

เมื่อจบ Lab นี้ ผู้เรียนจะสามารถ:

1. อธิบายสถาปัตยกรรมของระบบ RAG และองค์ประกอบหลักแต่ละส่วนได้
2. สร้างกระบวนการ Ingestion → Chunking → Embedding → Vector Store ได้ด้วยตนเอง
3. สร้างระบบ Retrieval ทั้งแบบ Dense, Sparse (BM25) และ Hybrid พร้อม Re-ranking
4. ต่อผลลัพธ์การค้นคืน (Retrieval) เข้ากับ Large Language Model เพื่อสร้างคำตอบ (Generation)
5. ประเมินคุณภาพของระบบ RAG ทั้งในมิติ Retrieval และ Generation
6. สร้าง Demo UI อย่างง่ายด้วย Gradio

## 🗺️ กำหนดการ (Agenda)

| ช่วงเวลา | หัวข้อ |
|---|---|
| 0:00 – 0:15 | Part 0 — Setup สภาพแวดล้อมและติดตั้งไลบรารี |
| 0:15 – 0:45 | Part 1 — Data Ingestion (โหลดเอกสาร) |
| 0:45 – 1:15 | Part 2 — Chunking (แบ่งเอกสารเป็นชิ้นย่อย) |
| 1:15 – 1:45 | Part 3 — Embeddings และ Vector Store |
| 1:45 – 2:15 | Part 4 — Retrieval (Dense / Sparse / Hybrid / Re-ranking) |
| 2:15 – 3:00 | Part 5 — Generation: ประกอบร่างเป็น RAG Pipeline ฉบับสมบูรณ์ |
| 3:00 – 3:20 | Part 6 — Evaluation (การประเมินผล) *(ส่วนเสริม หากเวลาเหลือ)* |
| 3:20 – 3:30 | Part 7 — สรุปและแบบฝึกหัดต่อยอด |

> ⚠️ **หมายเหตุด้านความถูกต้องของเนื้อหา:** ชุดข้อมูลตัวอย่างที่ใช้ใน Lab นี้เป็นข้อความสมมติที่สร้างขึ้นเพื่อการสอนเท่านั้น ไม่ใช่ข้อมูลข้อเท็จจริงของบริษัทหรือหน่วยงานใด


## 🧰 เทคโนโลยีที่ใช้ใน Lab นี้

Lab นี้ออกแบบให้ **รันได้ฟรีบน Google Colab โดยไม่ต้องใช้ API Key ของผู้ให้บริการเชิงพาณิชย์ใด ๆ** โดยใช้โมเดล Open-Source ทั้งหมด:

| องค์ประกอบ | เครื่องมือที่ใช้ | หมายเหตุ |
|---|---|---|
| Document Loading | `pypdf`, Python I/O | รองรับทั้งข้อความตัวอย่างและไฟล์ PDF ที่อัปโหลดเอง |
| Chunking | `langchain-text-splitters` | Recursive Character Text Splitter |
| Embedding Model | `sentence-transformers` (`all-MiniLM-L6-v2`) | โมเดลเปิด ขนาดเล็ก เร็ว |
| Vector Store | `faiss-cpu` (Facebook AI Similarity Search) | ค้นคืนแบบ Dense |
| Sparse Retrieval | `rank_bm25` | ค้นคืนแบบ Keyword-based |
| Re-ranking | `sentence-transformers` `CrossEncoder` | ปรับลำดับผลลัพธ์ให้แม่นยำขึ้น |
| Generation (LLM) | `transformers` (`google/flan-t5-base`) | โมเดลเปิด ไม่ต้องขอสิทธิ์เข้าถึง |
| Demo UI | `gradio` | สร้างหน้าเว็บสาธิตอย่างง่าย |

**ทางเลือกขั้นสูง (Optional):** หากมี API Key ของ OpenAI หรือ Google Gemini ผู้สอนสามารถสลับไปใช้โมเดลเชิงพาณิชย์แทนในขั้นตอน Generation ได้ (มีตัวอย่างโค้ดคอมเมนต์ไว้ให้ในภายหลัง) เพื่อให้ได้คำตอบที่มีคุณภาพสูงขึ้น


---
# ⚙️ Part 0 — Setup สภาพแวดล้อม (15 นาที)

**ขั้นตอนก่อนเริ่ม:**
1. ไปที่เมนู `Runtime` → `Change runtime type`
2. เลือก Hardware accelerator เป็น **T4 GPU** (หากมี Quota) หรือ **CPU** ก็สามารถรันได้เช่นกัน (ช้ากว่าในขั้นตอน Generation เล็กน้อย)
3. กด `Save` แล้วรันเซลล์ด้านล่างตามลำดับ


In [ ]:
# ตรวจสอบว่าเชื่อมต่อ GPU หรือไม่
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("กำลังรันด้วย CPU — Lab นี้ยังคงทำงานได้ปกติ เพียงแต่ขั้นตอน Generation จะช้าลงเล็กน้อย")


In [ ]:
# ติดตั้งไลบรารีที่จำเป็นทั้งหมด (ใช้เวลาประมาณ 1-2 นาที)
!pip install -q sentence-transformers faiss-cpu transformers accelerate \
    langchain langchain-community langchain-text-splitters pypdf \
    rank_bm25 gradio scikit-learn matplotlib
print("ติดตั้งไลบรารีเรียบร้อยแล้ว ✅")


In [ ]:
import os
import re
import time
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
from rank_bm25 import BM25Okapi
from transformers import pipeline

import warnings
warnings.filterwarnings("ignore")

print("นำเข้าไลบรารีเรียบร้อยแล้ว ✅")


---
# 📄 Part 1 — Data Ingestion (30 นาที)

RAG เริ่มต้นด้วยการ**นำเอกสารเข้าสู่ระบบ** (Ingestion) ในตัวอย่างนี้เราจะใช้คลังความรู้สมมติ
เกี่ยวกับ "บริษัท Aurora Cloud" (ข้อมูลสมมติทั้งหมด สร้างขึ้นเพื่อการสอน) และเปิดโอกาสให้ผู้เรียน
อัปโหลดเอกสาร PDF ของตนเองเพื่อทดลองเพิ่มเติม


In [ ]:
# ชุดข้อมูลตัวอย่าง (Sample Knowledge Base)
# หมายเหตุ: เนื้อหาทั้งหมดเป็นข้อมูลสมมติที่สร้างขึ้นเพื่อจุดประสงค์ทางการศึกษาเท่านั้น

documents = [
    {
        "id": "doc_1",
        "title": "นโยบายการลาพักร้อน",
        "text": """บริษัท Aurora Cloud (สมมติ) กำหนดให้พนักงานประจำมีสิทธิ์ลาพักร้อนได้ปีละ 15 วันทำการ
        โดยพนักงานที่ทำงานครบ 1 ปีขึ้นไปจะได้รับสิทธิ์ลาพักร้อนเพิ่มอีก 1 วันต่อปีที่ทำงาน สูงสุดไม่เกิน 20 วันต่อปี
        การขอลาพักร้อนต้องแจ้งล่วงหน้าอย่างน้อย 3 วันทำการผ่านระบบ HR Portal และต้องได้รับการอนุมัติจากหัวหน้างานโดยตรง
        วันลาพักร้อนที่ใช้ไม่หมดในปีนั้นสามารถสะสมยกไปปีถัดไปได้ไม่เกิน 5 วัน"""
    },
    {
        "id": "doc_2",
        "title": "นโยบายการทำงานจากที่บ้าน (Work From Home)",
        "text": """พนักงานของ Aurora Cloud (สมมติ) สามารถทำงานจากที่บ้านได้สูงสุด 2 วันต่อสัปดาห์
        โดยต้องแจ้งหัวหน้างานล่วงหน้าอย่างน้อย 1 วัน ตำแหน่งงานที่ต้องปฏิบัติงานหน้างาน เช่น ฝ่ายซ่อมบำรุงหรือฝ่ายต้อนรับ
        ไม่สามารถขอทำงานจากที่บ้านได้ พนักงานที่ทำงานจากที่บ้านต้องเข้าร่วมประชุมทีมผ่านวิดีโอคอลตามเวลาที่กำหนด
        และต้องตอบกลับข้อความในช่องทางการสื่อสารหลักภายใน 2 ชั่วโมงในเวลาทำงาน"""
    },
    {
        "id": "doc_3",
        "title": "สวัสดิการด้านสุขภาพ",
        "text": """Aurora Cloud (สมมติ) มอบประกันสุขภาพกลุ่มให้พนักงานประจำทุกคนตั้งแต่วันแรกที่เริ่มงาน
        ครอบคลุมค่ารักษาพยาบาลผู้ป่วยในและผู้ป่วยนอก วงเงินสูงสุด 200,000 บาทต่อปี
        พนักงานสามารถเพิ่มความคุ้มครองให้คู่สมรสและบุตรได้โดยชำระเบี้ยประกันส่วนเพิ่มเอง
        นอกจากนี้บริษัทยังสนับสนุนค่าตรวจสุขภาพประจำปีให้พนักงานทุกคนปีละ 1 ครั้ง"""
    },
    {
        "id": "doc_4",
        "title": "กระบวนการเบิกค่าใช้จ่าย (Expense Reimbursement)",
        "text": """การเบิกค่าใช้จ่ายของ Aurora Cloud (สมมติ) ต้องยื่นผ่านระบบ Finance Portal พร้อมแนบใบเสร็จรับเงินตัวจริง
        ภายใน 30 วันนับจากวันที่เกิดค่าใช้จ่าย ค่าใช้จ่ายที่เกิน 5,000 บาทต้องได้รับอนุมัติล่วงหน้าจากผู้จัดการฝ่ายก่อนดำเนินการ
        ระยะเวลาการอนุมัติและโอนเงินคืนอยู่ที่ประมาณ 7-10 วันทำการหลังจากยื่นเอกสารครบถ้วน"""
    },
    {
        "id": "doc_5",
        "title": "นโยบายความปลอดภัยของข้อมูล (Data Security Policy)",
        "text": """พนักงาน Aurora Cloud (สมมติ) ทุกคนต้องเปิดใช้งาน Two-Factor Authentication (2FA) กับบัญชีงานทุกบัญชี
        ห้ามนำข้อมูลลูกค้าออกจากระบบภายในโดยเด็ดขาด และห้ามใช้ USB Drive ส่วนตัวเชื่อมต่อกับเครื่องคอมพิวเตอร์ของบริษัท
        พนักงานต้องเปลี่ยนรหัสผ่านทุก 90 วัน และต้องผ่านการอบรมด้านความปลอดภัยไซเบอร์ปีละ 1 ครั้ง
        การละเมิดนโยบายนี้อาจมีผลต่อการพิจารณาทางวินัยตามระเบียบบริษัท"""
    },
    {
        "id": "doc_6",
        "title": "กระบวนการ Onboarding พนักงานใหม่",
        "text": """พนักงานใหม่ของ Aurora Cloud (สมมติ) จะได้รับอุปกรณ์ทำงาน (โน้ตบุ๊กและอุปกรณ์เสริม) ภายในวันแรกที่เริ่มงาน
        สัปดาห์แรกจะเป็นการปฐมนิเทศร่วมกับฝ่ายบุคคลและหัวหน้างานโดยตรง พนักงานใหม่ทุกคนต้องผ่านการอบรม
        หลักสูตรพื้นฐาน 3 หลักสูตร ได้แก่ วัฒนธรรมองค์กร ความปลอดภัยของข้อมูล และเครื่องมือการทำงานภายใน 30 วันแรก"""
    },
]

print(f"โหลดเอกสารตัวอย่างทั้งหมด {len(documents)} ฉบับ")
for d in documents:
    print(f"  - [{d['id']}] {d['title']} ({len(d['text'])} ตัวอักษร)")


## 📤 (ทางเลือก) อัปโหลดไฟล์ PDF ของตัวเอง

รันเซลล์ด้านล่างเพื่ออัปโหลดไฟล์ PDF แล้วระบบจะดึงข้อความออกมาต่อท้ายชุดข้อมูล `documents`
ขั้นตอนนี้ข้ามได้หากต้องการใช้ชุดข้อมูลตัวอย่างต่อไปก่อน


In [ ]:
from pypdf import PdfReader

def load_pdf_as_document(filepath):
    reader = PdfReader(filepath)
    text = "\n".join(page.extract_text() or "" for page in reader.pages)
    return text

UPLOAD_OWN_PDF = False  # เปลี่ยนเป็น True หากต้องการอัปโหลดไฟล์ของตนเอง

if UPLOAD_OWN_PDF:
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        text = load_pdf_as_document(fname)
        documents.append({"id": fname, "title": fname, "text": text})
        print(f"เพิ่มเอกสาร '{fname}' แล้ว ({len(text)} ตัวอักษร)")
else:
    print("ข้ามขั้นตอนนี้ — ใช้ชุดข้อมูลตัวอย่างต่อไป (ตั้ง UPLOAD_OWN_PDF = True เพื่อเปิดใช้งาน)")


**🧪 Checkpoint 1 — คำถามชวนคิด**

- เหตุใดขั้นตอน Ingestion จึงสำคัญต่อคุณภาพของระบบ RAG โดยรวม?
- หากเอกสารต้นทางมีคุณภาพต่ำ (เช่น สแกนไม่ชัด, จัดรูปแบบไม่ดี) จะส่งผลต่อขั้นตอนถัดไปอย่างไร?


---
# ✂️ Part 2 — Chunking: การแบ่งเอกสารเป็นชิ้นย่อย (30 นาที)

โมเดล Embedding และ LLM มีข้อจำกัดด้านความยาว Context ดังนั้นเราจึงต้องแบ่งเอกสารยาว ๆ
ออกเป็น "ชิ้น" (Chunk) ที่มีขนาดพอเหมาะ พร้อมกำหนดค่า **Overlap** เพื่อไม่ให้ใจความสำคัญขาดตอนระหว่างรอยต่อ


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,       # จำนวนตัวอักษรต่อ 1 chunk (ปรับได้)
    chunk_overlap=40,     # จำนวนตัวอักษรที่ overlap กันระหว่าง chunk
    separators=["\n\n", "\n", "。", ".", " ", ""],
)

chunks = []
for doc in documents:
    splits = splitter.split_text(doc["text"])
    for i, chunk_text in enumerate(splits):
        chunks.append({
            "chunk_id": f"{doc['id']}_chunk_{i}",
            "doc_id": doc["id"],
            "title": doc["title"],
            "text": chunk_text.strip(),
        })

print(f"แบ่งเอกสาร {len(documents)} ฉบับ ออกเป็นทั้งหมด {len(chunks)} chunks")
print("\nตัวอย่าง 3 chunk แรก:")
for c in chunks[:3]:
    print(f"\n[{c['chunk_id']}]")
    print(textwrap.fill(c["text"], width=90))


## 🔬 ทดลอง: เปรียบเทียบขนาด Chunk ที่แตกต่างกัน

ลองปรับค่า `chunk_size` และ `chunk_overlap` แล้วสังเกตผลกระทบต่อจำนวน chunk ที่ได้


In [ ]:
chunk_size_options = [100, 200, 400, 800]
results = []

for size in chunk_size_options:
    test_splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=int(size * 0.2))
    total_chunks = sum(len(test_splitter.split_text(d["text"])) for d in documents)
    results.append({"chunk_size": size, "overlap": int(size * 0.2), "total_chunks": total_chunks})

df_compare = pd.DataFrame(results)
print(df_compare)

plt.figure(figsize=(6, 4))
plt.bar(df_compare["chunk_size"].astype(str), df_compare["total_chunks"], color="#1C7293")
plt.xlabel("Chunk Size (characters)")
plt.ylabel("จำนวน Chunks ทั้งหมด")
plt.title("ผลของขนาด Chunk ต่อจำนวน Chunk ที่ได้")
plt.tight_layout()
plt.show()


**📝 แบบฝึกหัด 2.1:** ลองปรับ `chunk_size=150` และ `chunk_overlap=30` ในเซลล์ Chunking ด้านบน
แล้วสังเกตว่าเนื้อหาของ 3 chunk แรกเปลี่ยนไปอย่างไร มีข้อความใดถูกตัดขาดใจความหรือไม่?

**🧪 Checkpoint 2 — คำถามชวนคิด**

- Chunk ขนาดเล็กเกินไป vs. ขนาดใหญ่เกินไป มีข้อดี-ข้อเสียต่างกันอย่างไร?
- เพราะเหตุใดจึงต้องมี Overlap ระหว่าง Chunk?


---
# 🧠 Part 3 — Embeddings และ Vector Store (30 นาที)

**Embedding** คือการแปลงข้อความให้เป็นเวกเตอร์ตัวเลขในปริภูมิหลายมิติ โดยข้อความที่มีความหมายใกล้เคียงกัน
จะมีเวกเตอร์อยู่ใกล้กันในปริภูมินั้น เราจะใช้โมเดล `all-MiniLM-L6-v2` จาก Sentence-Transformers
ซึ่งเป็นโมเดลขนาดเล็ก เร็ว และให้ผลลัพธ์ดีเพียงพอสำหรับงานสอน


In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    normalize_embeddings=True,   # normalize เพื่อให้ใช้ cosine similarity ผ่าน inner product ได้
)

print(f"สร้าง embedding สำเร็จ: shape = {chunk_embeddings.shape}")
print(f"มิติของ embedding แต่ละตัว: {chunk_embeddings.shape[1]}")


In [ ]:
# สร้าง FAISS Vector Index (ใช้ Inner Product บน embedding ที่ normalize แล้ว = cosine similarity)
embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(np.array(chunk_embeddings).astype("float32"))

print(f"สร้าง FAISS index สำเร็จ: จำนวนเวกเตอร์ทั้งหมด = {index.ntotal}")


## 📊 (ทางเลือก) การมองเห็น Embedding ในรูปแบบ 2 มิติ

เราจะลดมิติของ embedding จาก 384 มิติ เหลือ 2 มิติด้วย PCA เพื่อลองมองภาพรวมว่า Chunk ใดอยู่ใกล้กันบ้าง


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
reduced = pca.fit_transform(chunk_embeddings)

doc_ids_unique = list({c["doc_id"] for c in chunks})
colors = plt.cm.tab10(np.linspace(0, 1, len(doc_ids_unique)))
color_map = dict(zip(doc_ids_unique, colors))

plt.figure(figsize=(7, 6))
for i, c in enumerate(chunks):
    plt.scatter(reduced[i, 0], reduced[i, 1], color=color_map[c["doc_id"]], s=60)
    plt.annotate(c["doc_id"].split("_")[1], (reduced[i, 0], reduced[i, 1]), fontsize=8)

plt.title("การกระจายตัวของ Chunk Embeddings (ลดมิติด้วย PCA)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.tight_layout()
plt.show()


**📝 แบบฝึกหัด 3.1 (ขั้นสูง):** ลองเปลี่ยนโมเดล embedding เป็น `BAAI/bge-small-en-v1.5`
หรือ `intfloat/multilingual-e5-small` (รองรับภาษาไทยดีขึ้น) แล้วเปรียบเทียบผลลัพธ์การค้นคืนในหัวข้อถัดไป

**🧪 Checkpoint 3 — คำถามชวนคิด**

- เพราะเหตุใดเราจึง normalize embedding ก่อนสร้าง FAISS index?
- Embedding model ที่เทรนด้วยข้อมูลภาษาอังกฤษเป็นหลัก จะมีข้อจำกัดอย่างไรเมื่อใช้กับข้อความภาษาไทย?


---
# 🔍 Part 4 — Retrieval: Dense, Sparse และ Hybrid (30 นาที)

ในส่วนนี้เราจะสร้างฟังก์ชันการค้นคืน 3 รูปแบบ:

1. **Dense Retrieval** — ค้นคืนด้วยความคล้ายคลึงเชิงความหมาย (Semantic Similarity) ผ่าน FAISS
2. **Sparse Retrieval (BM25)** — ค้นคืนแบบอิงคำสำคัญ (Keyword-based)
3. **Hybrid + Re-ranking** — ผสานทั้งสองแบบ แล้วจัดลำดับใหม่ด้วย Cross-Encoder


In [ ]:
def dense_retrieve(query, top_k=3):
    query_vec = embedding_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(query_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({**chunks[idx], "score": float(score)})
    return results


# ทดสอบ Dense Retrieval
test_query = "พนักงานลาพักร้อนได้กี่วันต่อปี"
results = dense_retrieve(test_query, top_k=3)

print(f"คำถาม: {test_query}\n")
for r in results:
    print(f"[score={r['score']:.3f}] ({r['title']})")
    print(textwrap.fill(r["text"], width=90))
    print()


In [ ]:
# Sparse Retrieval ด้วย BM25 (อิงการตัดคำง่าย ๆ ด้วย whitespace/regex)
def simple_tokenize(text):
    return re.findall(r"[\w']+", text.lower())

tokenized_corpus = [simple_tokenize(c["text"]) for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

def sparse_retrieve(query, top_k=3):
    tokenized_query = simple_tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [{**chunks[i], "score": float(scores[i])} for i in top_indices]


results_sparse = sparse_retrieve(test_query, top_k=3)
print(f"คำถาม: {test_query}\n")
for r in results_sparse:
    print(f"[score={r['score']:.3f}] ({r['title']})")
    print(textwrap.fill(r["text"], width=90))
    print()


In [ ]:
# Hybrid Retrieval: รวมผลจาก Dense + Sparse ด้วยการถ่วงน้ำหนัก (Reciprocal Rank Fusion แบบง่าย)
def hybrid_retrieve(query, top_k=5, alpha=0.5):
    dense_results = dense_retrieve(query, top_k=len(chunks))
    sparse_results = sparse_retrieve(query, top_k=len(chunks))

    def normalize(results):
        scores = np.array([r["score"] for r in results])
        if scores.max() - scores.min() > 1e-9:
            norm = (scores - scores.min()) / (scores.max() - scores.min())
        else:
            norm = np.zeros_like(scores)
        return {r["chunk_id"]: n for r, n in zip(results, norm)}

    dense_scores = normalize(dense_results)
    sparse_scores = normalize(sparse_results)

    combined = {}
    for c in chunks:
        cid = c["chunk_id"]
        combined[cid] = alpha * dense_scores.get(cid, 0) + (1 - alpha) * sparse_scores.get(cid, 0)

    ranked_ids = sorted(combined, key=combined.get, reverse=True)[:top_k]
    chunk_lookup = {c["chunk_id"]: c for c in chunks}
    return [{**chunk_lookup[cid], "score": combined[cid]} for cid in ranked_ids]


results_hybrid = hybrid_retrieve(test_query, top_k=3)
print(f"คำถาม: {test_query}\n")
for r in results_hybrid:
    print(f"[hybrid_score={r['score']:.3f}] ({r['title']})")
    print(textwrap.fill(r["text"], width=90))
    print()


In [ ]:
# Re-ranking ด้วย Cross-Encoder (แม่นยำกว่าการคำนวณ similarity แบบตรง แต่ช้ากว่า
# จึงมักใช้ "รอบสอง" กับผลลัพธ์จำนวนน้อยที่ผ่านการคัดกรองมาแล้ว)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, top_k=3):
    pairs = [[query, c["text"]] for c in candidates]
    ce_scores = cross_encoder.predict(pairs)
    for c, s in zip(candidates, ce_scores):
        c["rerank_score"] = float(s)
    return sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:top_k]


candidates = hybrid_retrieve(test_query, top_k=6)
reranked = rerank(test_query, candidates, top_k=3)

print(f"คำถาม: {test_query}\n")
for r in reranked:
    print(f"[rerank_score={r['rerank_score']:.3f}] ({r['title']})")
    print(textwrap.fill(r["text"], width=90))
    print()


**📝 แบบฝึกหัด 4.1:** ลองเปลี่ยน `test_query` เป็นคำถามอื่น ๆ เช่น
`"บริษัทมีสวัสดิการประกันสุขภาพอะไรบ้าง"` หรือ `"ต้องเปลี่ยนรหัสผ่านบ่อยแค่ไหน"`
แล้วเปรียบเทียบผลลัพธ์จากทั้ง 3 วิธี ว่าให้คำตอบสอดคล้องกันหรือไม่

**🧪 Checkpoint 4 — คำถามชวนคิด**

- ในสถานการณ์ใดที่ BM25 (Sparse) จะทำงานได้ดีกว่า Dense Retrieval?
- เหตุใดการทำ Re-ranking จึงมักทำกับผลลัพธ์แค่ 5-10 อันดับแรก แทนที่จะทำกับเอกสารทั้งหมด?


---
# 🤖 Part 5 — Generation: ประกอบร่างเป็น RAG Pipeline ฉบับสมบูรณ์ (45 นาที)

ถึงขั้นตอนสำคัญที่สุด: นำผลลัพธ์จากการค้นคืน (Retrieval) มาใส่ใน Prompt แล้วส่งให้ Language Model
สร้างคำตอบสุดท้าย โดยในที่นี้จะใช้โมเดลเปิด `google/flan-t5-base` ซึ่งไม่ต้องขอ API Key ใด ๆ


In [ ]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=0 if torch.cuda.is_available() else -1,
)
print("โหลดโมเดลสำหรับ Generation สำเร็จ ✅")


### 🔁 ทางเลือก: ใช้โมเดลเชิงพาณิชย์แทน (OpenAI / Google Gemini)

หากมี API Key และต้องการคำตอบคุณภาพสูงขึ้น สามารถแทนที่ฟังก์ชัน `generate_answer()`
ในเซลล์ถัดไปด้วยโค้ดตัวอย่างนี้ (ไม่บังคับใช้งาน):

```python
# ตัวอย่าง (ไม่ต้องรัน หากไม่มี API Key):
# from openai import OpenAI
# client = OpenAI(api_key="YOUR_API_KEY")
#
# def generate_answer_openai(prompt):
#     response = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[{"role": "user", "content": prompt}],
#         temperature=0.2,
#     )
#     return response.choices[0].message.content
```


In [ ]:
RAG_PROMPT_TEMPLATE = """ตอบคำถามโดยอ้างอิงจาก "บริบท" ที่ให้มาเท่านั้น
หากไม่มีข้อมูลเพียงพอในบริบท ให้ตอบว่า "ไม่มีข้อมูลเพียงพอในเอกสารที่ให้มา"
ห้ามสร้างข้อมูลขึ้นเองนอกเหนือจากบริบท

บริบท:
{context}

คำถาม: {question}

คำตอบ:"""


def build_prompt(question, retrieved_chunks):
    context = "\n\n".join(
        f"[{c['title']}]\n{c['text']}" for c in retrieved_chunks
    )
    return RAG_PROMPT_TEMPLATE.format(context=context, question=question)


def generate_answer(prompt, max_new_tokens=200):
    output = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    return output[0]["generated_text"]


print("กำหนดฟังก์ชัน Prompt Template และ Generation เรียบร้อยแล้ว ✅")


In [ ]:
def rag_pipeline(question, top_k=3, use_reranker=True, verbose=True):
    """
    RAG pipeline แบบครบวงจร:
    1) Hybrid retrieve ผู้สมัคร (candidates)
    2) (ทางเลือก) Re-rank ด้วย Cross-Encoder
    3) สร้าง Prompt จาก context ที่ค้นเจอ
    4) ส่งให้ LLM สร้างคำตอบ
    """
    candidates = hybrid_retrieve(question, top_k=top_k * 2)
    retrieved = rerank(question, candidates, top_k=top_k) if use_reranker else candidates[:top_k]

    prompt = build_prompt(question, retrieved)
    answer = generate_answer(prompt)

    if verbose:
        print(f"❓ คำถาม: {question}\n")
        print("📚 เอกสารที่ค้นเจอ (Sources):")
        for r in retrieved:
            print(f"   - {r['title']}")
        print(f"\n💬 คำตอบจาก RAG:\n{answer}\n")
        print("-" * 80)

    return {"question": question, "answer": answer, "sources": retrieved}


# ทดสอบ Pipeline แบบครบวงจร
test_questions = [
    "พนักงานลาพักร้อนได้กี่วันต่อปี",
    "ทำงานจากที่บ้านได้สัปดาห์ละกี่วัน",
    "การเบิกค่าใช้จ่ายต้องยื่นภายในกี่วัน",
]

for q in test_questions:
    rag_pipeline(q)


## ⚖️ เปรียบเทียบ: มี RAG vs. ไม่มี RAG

เพื่อให้เห็นคุณค่าของ RAG อย่างชัดเจน ลองเปรียบเทียบคำตอบของ LLM **ที่ไม่มีบริบทใด ๆ**
กับคำตอบที่ได้จาก RAG Pipeline ในคำถามเดียวกัน


In [ ]:
def generate_without_rag(question):
    prompt = f"ตอบคำถามต่อไปนี้: {question}"
    return generate_answer(prompt)


compare_question = "นโยบายการลาพักร้อนของบริษัท Aurora Cloud เป็นอย่างไร"

print("=" * 80)
print("🚫 ไม่มี RAG (ตอบจากความรู้ในตัวโมเดลเท่านั้น):")
print(generate_without_rag(compare_question))

print("\n" + "=" * 80)
print("✅ มี RAG (ตอบโดยอ้างอิงจากเอกสารที่ค้นคืนมา):")
rag_pipeline(compare_question)


**📝 แบบฝึกหัด 5.1:** สังเกตความแตกต่างของคำตอบทั้งสองแบบ — คำตอบแบบไม่มี RAG น่าเชื่อถือหรือไม่?
มีการ "เดา" หรือสร้างข้อมูลขึ้นเอง (Hallucination) หรือไม่?

**📝 แบบฝึกหัด 5.2 (ขั้นสูง):** ลองปรับ `RAG_PROMPT_TEMPLATE` ให้โมเดลตอบเป็นภาษาอังกฤษ
หรือปรับให้ตอบในรูปแบบ bullet point แทนย่อหน้า

**🧪 Checkpoint 5 — คำถามชวนคิด**

- เหตุใดการกำหนดใน Prompt ว่า "หากไม่มีข้อมูลเพียงพอ ให้ตอบว่าไม่ทราบ" จึงสำคัญมากต่อการลด Hallucination?
- โมเดลขนาดเล็กอย่าง flan-t5-base มีข้อจำกัดใดบ้างเมื่อเทียบกับโมเดลเชิงพาณิชย์ขนาดใหญ่?


---
# 📊 Part 6 — Evaluation: การประเมินผลระบบ RAG (ส่วนเสริม หากเวลาเหลือ)

การประเมินระบบ RAG แบ่งออกเป็น 2 มิติหลัก:

1. **Retrieval Quality** — ระบบค้นคืนเอกสารที่ "ถูกต้อง" หรือไม่ (Precision, Recall, MRR)
2. **Generation Quality** — คำตอบสุดท้าย "ซื่อตรง" ต่อบริบท (Faithfulness) และ "ตรงประเด็น"
   กับคำถาม (Answer Relevancy) หรือไม่


In [ ]:
# ตัวอย่างชุดข้อมูลทดสอบอย่างง่าย (Query -> เอกสารที่ถูกต้อง)
eval_set = [
    {"query": "พนักงานลาพักร้อนได้กี่วันต่อปี", "relevant_doc_id": "doc_1"},
    {"query": "ทำงานจากที่บ้านได้สัปดาห์ละกี่วัน", "relevant_doc_id": "doc_2"},
    {"query": "บริษัทให้ประกันสุขภาพวงเงินเท่าไหร่", "relevant_doc_id": "doc_3"},
    {"query": "เบิกค่าใช้จ่ายต้องแนบเอกสารอะไร", "relevant_doc_id": "doc_4"},
    {"query": "ต้องเปิด 2FA หรือไม่", "relevant_doc_id": "doc_5"},
]

def precision_at_k(retrieved_chunks, relevant_doc_id, k=3):
    top_k = retrieved_chunks[:k]
    hits = sum(1 for c in top_k if c["doc_id"] == relevant_doc_id)
    return hits / k

def reciprocal_rank(retrieved_chunks, relevant_doc_id):
    for i, c in enumerate(retrieved_chunks):
        if c["doc_id"] == relevant_doc_id:
            return 1 / (i + 1)
    return 0.0

precisions, rr_scores = [], []
for item in eval_set:
    retrieved = hybrid_retrieve(item["query"], top_k=5)
    p = precision_at_k(retrieved, item["relevant_doc_id"], k=3)
    rr = reciprocal_rank(retrieved, item["relevant_doc_id"])
    precisions.append(p)
    rr_scores.append(rr)
    print(f"Query: {item['query']:<45} Precision@3={p:.2f}  RR={rr:.2f}")

print(f"\nค่าเฉลี่ย Precision@3 = {np.mean(precisions):.3f}")
print(f"Mean Reciprocal Rank (MRR) = {np.mean(rr_scores):.3f}")


## 🧪 การตรวจสอบ Faithfulness แบบง่าย (Heuristic)

ในการใช้งานจริงมักใช้เฟรมเวิร์กอย่าง **RAGAS**, **TruLens** หรือ **DeepEval** ซึ่งมักต้องใช้ LLM
(เช่น GPT-4) เป็นตัวตัดสิน (LLM-as-a-judge) — ในที่นี้จะสาธิตแนวคิดแบบง่ายด้วยการ
ตรวจสอบว่าคำตอบมีคำสำคัญที่ปรากฏในบริบทที่ค้นคืนมาหรือไม่ (เป็นเพียงการประมาณคร่าว ๆ)


In [ ]:
def simple_faithfulness_check(answer, context_chunks):
    context_text = " ".join(c["text"] for c in context_chunks).lower()
    answer_tokens = set(simple_tokenize(answer))
    context_tokens = set(simple_tokenize(context_text))
    if not answer_tokens:
        return 0.0
    overlap = answer_tokens & context_tokens
    return len(overlap) / len(answer_tokens)


result = rag_pipeline("พนักงานลาพักร้อนได้กี่วันต่อปี", verbose=False)
faith_score = simple_faithfulness_check(result["answer"], result["sources"])
print(f"คำตอบ: {result['answer']}")
print(f"Faithfulness (heuristic, token-overlap) ≈ {faith_score:.2f}")
print("\n⚠️ หมายเหตุ: นี่เป็นเพียงตัวชี้วัดอย่างง่ายเพื่อการสอน ไม่ควรใช้แทนการประเมินด้วย")
print("   เฟรมเวิร์กมาตรฐาน เช่น RAGAS ในการใช้งานจริง")


### 🔧 ต่อยอด: RAGAS (ต้องใช้ API Key ของ LLM)

หากมี OpenAI API Key สามารถติดตั้งและใช้ RAGAS เพื่อประเมิน `faithfulness`, `answer_relevancy`,
`context_precision`, และ `context_recall` ได้อย่างเป็นระบบมากขึ้น:

```python
# ตัวอย่าง (ไม่บังคับรัน — ต้องใช้ API Key):
# !pip install -q ragas datasets
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
# ...
```

**🧪 Checkpoint 6 — คำถามชวนคิด**

- ทำไมการประเมินคุณภาพของ Generation จึงยากกว่าการประเมิน Retrieval?
- เพราะเหตุใด "LLM-as-a-judge" จึงเป็นแนวทางที่นิยมในการประเมิน RAG เชิงคุณภาพ (Faithfulness, Relevancy)?


---
# 🖥️ ส่วนเสริม — สร้าง Demo UI ด้วย Gradio

ลองสร้างหน้าเว็บสาธิตอย่างง่ายเพื่อให้ผู้ใช้พิมพ์คำถามและรับคำตอบจากระบบ RAG ที่สร้างขึ้น


In [ ]:
import gradio as gr

def gradio_rag_interface(question):
    result = rag_pipeline(question, verbose=False)
    sources_text = "\n".join(f"- {s['title']}" for s in result["sources"])
    return result["answer"], sources_text

demo = gr.Interface(
    fn=gradio_rag_interface,
    inputs=gr.Textbox(label="คำถามของคุณ", placeholder="เช่น พนักงานลาพักร้อนได้กี่วันต่อปี"),
    outputs=[gr.Textbox(label="คำตอบ"), gr.Textbox(label="แหล่งอ้างอิง (Sources)")],
    title="🔎 RAG Demo — ระบบตอบคำถามจากคลังความรู้บริษัท (ข้อมูลสมมติ)",
    examples=[
        "พนักงานลาพักร้อนได้กี่วันต่อปี",
        "ทำงานจากที่บ้านได้สัปดาห์ละกี่วัน",
        "บริษัทให้ประกันสุขภาพวงเงินเท่าไหร่",
    ],
)

demo.launch(share=True, debug=False)


---
# 🏁 Part 7 — สรุปและแบบฝึกหัดต่อยอด

## ✅ สรุปสิ่งที่ได้เรียนรู้ใน Lab นี้

- โครงสร้างของ RAG Pipeline ทั้ง 6 ขั้นตอน: Ingestion → Chunking → Embedding → Vector Store → Retrieval → Generation
- ความแตกต่างระหว่าง Dense, Sparse (BM25) และ Hybrid Retrieval
- การทำ Re-ranking ด้วย Cross-Encoder เพื่อเพิ่มความแม่นยำ
- การออกแบบ Prompt เพื่อลด Hallucination และบังคับให้โมเดลอ้างอิงบริบทเท่านั้น
- แนวคิดพื้นฐานของการประเมินผลระบบ RAG ทั้งด้าน Retrieval และ Generation

## 🚀 แบบฝึกหัดต่อยอด (Take-home Challenges)

1. **เปลี่ยนชุดข้อมูล:** นำเอกสารจริงของหน่วยงาน/สาขาวิชาที่สนใจมาสร้างเป็น RAG ของตนเอง
2. **ทดลองโมเดล Embedding หลายตัว:** เปรียบเทียบ `all-MiniLM-L6-v2`, `bge-small-en-v1.5`,
   และ `multilingual-e5-small` ว่าโมเดลใดค้นคืนภาษาไทยได้แม่นยำกว่ากัน
3. **ทดลอง Chunking แบบ Semantic:** ค้นคว้าเทคนิค Semantic Chunking (แบ่งตามความหมายแทนจำนวนตัวอักษร)
4. **เพิ่มเทคนิค Query Transformation:** ลองเขียนฟังก์ชัน HyDE (Hypothetical Document Embeddings)
   หรือ Multi-Query Retrieval
5. **เปลี่ยนไปใช้ Vector Database ระดับ Production:** ทดลองใช้ Chroma, Qdrant หรือ pgvector แทน FAISS
6. **เชื่อมต่อ LLM คุณภาพสูงขึ้น:** ทดลองใช้ Qwen2.5-1.5B-Instruct (ต้องใช้ GPU) หรือ API เชิงพาณิชย์
7. **ประเมินผลอย่างเป็นระบบ:** ติดตั้งและใช้งาน RAGAS เพื่อวัด Faithfulness และ Answer Relevancy อย่างจริงจัง

## 📚 แหล่งข้อมูลเพิ่มเติม

- Lewis, P., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.*
  arXiv:2005.11401
- เอกสารทางการของ LangChain: https://python.langchain.com/
- เอกสารทางการของ LlamaIndex: https://docs.llamaindex.ai/
- เอกสารทางการของ Sentence-Transformers: https://www.sbert.net/
- เอกสารทางการของ FAISS: https://faiss.ai/
- เอกสารทางการของ RAGAS: https://docs.ragas.io/

---
**🎉 ขอแสดงความยินดี! คุณได้สร้างระบบ RAG แบบครบวงจรด้วยตนเองเรียบร้อยแล้ว**
